Ce script implémente un système de classification d'images satellites pour prédire les niveaux de richesse en Afrique en utilisant VGG16 avec toutes ses couches dégelées (entraînables). Il utilise le transfer learning en partant d'un modèle VGG16 pré-entraîné sur ImageNet, mais sans geler aucune couche, permettant ainsi à tout le réseau d'être affiné sur la tâche spécifique de classification en 4 classes de richesse.
Une gestion robuste des données est implémentée via la classe ImageDataset, qui gère les cas d'images corrompues ou manquantes en les remplaçant par des tenseurs nuls (label -1). Les images sont normalisées et redimensionnées à 224x224 pixels selon les standards ImageNet. Le dataset est divisé en 80% pour l'entraînement et 20% pour la validation.
L'entraînement utilise SGD comme optimiseur avec un taux d'apprentissage de 0.001 et un momentum de 0.9, ainsi que la cross-entropy comme fonction de perte. Un système d'early stopping est implémenté avec une patience de 7 époques pour éviter le surapprentissage. Le meilleur modèle est sauvegardé automatiquement lorsque la perte de validation s'améliore.
Le script inclut des visualisations détaillées (distribution des classes, courbes d'apprentissage) et des métriques d'évaluation complètes (matrice de confusion, rapport de classification). Les résultats, incluant l'état du modèle, l'historique d'entraînement et les métriques d'évaluation, sont sauvegardés dans un fichier unique avec le suffixe "couche_nongelee" pour indiquer que toutes les couches sont entraînables.

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

# Configuration
batch_size = 16
learning_rate = 0.001
momentum = 0.9
num_epochs = 100
patience = 7

# Chemins vers les fichiers et répertoires
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\Data_Africa_sampled_with_images.csv"
image_folder = r"D:\wealth_predict_sentinel\Data\downloaded\Image_satellite_Divers_Pays_Afrique_Zoom_14_Sentinel_2_an_2023"
models_path = r"D:\wealth_predict_sentinel\models"
os.makedirs(models_path, exist_ok=True)

# Fonction pour visualiser la distribution des classes
def plot_class_distribution(df, title):
    counts = Counter(df['class'])
    plt.figure(figsize=(8, 6))
    plt.bar(counts.keys(), counts.values(), color=['blue', 'orange', 'green', 'red'])
    plt.title(title)
    plt.xlabel('Classe')
    plt.ylabel('Nombre')
    plt.show()

# Fonction pour afficher les statistiques des classes
def count_classes(df, name):
    counts = df['class'].value_counts()
    print(f"Répartition des classes dans {name} :")
    for cls, count in counts.items():
        print(f"  Classe {cls} : {count} observations")
    print("-" * 40)

# Dataset personnalisé avec gestion d'erreurs
class ImageDataset(Dataset):
    def __init__(self, dataframe, data_path, transform=None):
        self.dataframe = dataframe
        self.data_path = data_path
        self.transform = transform
        self.default_tensor = self.get_default_tensor()
        self.validate_images()

    def get_default_tensor(self):
        dummy_tensor = torch.zeros(3, 224, 224)
        return dummy_tensor

    def validate_images(self):
        valid_indices = []
        for idx in range(len(self.dataframe)):
            img_path = os.path.join(self.data_path, self.dataframe.iloc[idx]['Image_Name'])
            try:
                with Image.open(img_path) as img:
                    img.convert("RGB")
                valid_indices.append(idx)
            except Exception as e:
                print(f"Warning: Impossible de charger l'image {img_path}: {str(e)}")
        
        self.dataframe = self.dataframe.iloc[valid_indices].reset_index(drop=True)
        print(f"Nombre d'images valides chargées: {len(valid_indices)}")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            img_name = self.dataframe.iloc[idx]['Image_Name']
            img_path = os.path.join(self.data_path, img_name)
            
            with Image.open(img_path) as img:
                image = img.convert("RGB")
                
            if self.transform:
                image = self.transform(image)
            
            label = self.dataframe.iloc[idx]['class']
            return image, label
            
        except Exception as e:
            print(f"Warning: Utilisation d'un tenseur par défaut pour l'image {img_path}: {str(e)}")
            return self.default_tensor, -1

# Modèle VGG16 sans couches gelées
class VGG16Standard(nn.Module):
    def __init__(self, num_classes=4):
        super(VGG16Standard, self).__init__()
        self.vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        
        # Modifier uniquement la dernière couche pour notre nombre de classes
        num_features = self.vgg16.classifier[-1].in_features
        self.vgg16.classifier[-1] = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.vgg16(x)

def train_model_with_early_stopping(model, train_loader, val_loader, criterion, optimizer, num_epochs, patience, device):
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(num_epochs):
        print(f"\nÉpoque {epoch+1}/{num_epochs}")
        
        # Phase d'entraînement
        model.train()
        total_train_loss = 0
        train_predictions = []
        train_labels_list = []

        for images, labels in tqdm(train_loader, desc="Entraînement"):
            try:
                valid_mask = labels != -1
                if not valid_mask.any():
                    continue
                    
                images = images[valid_mask].to(device)
                labels = labels[valid_mask].to(device)
                labels = labels.long()  # Conversion des étiquettes en LongTensor
                
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                total_train_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                train_predictions.extend(preds.cpu().numpy())
                train_labels_list.extend(labels.cpu().numpy())
                
            except Exception as e:
                print(f"Erreur lors du traitement du batch: {str(e)}")
                continue

        avg_train_loss = total_train_loss / len(train_loader)
        train_accuracy = accuracy_score(train_labels_list, train_predictions)
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        # Phase de validation
        model.eval()
        total_val_loss = 0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                try:
                    valid_mask = labels != -1
                    if not valid_mask.any():
                        continue
                        
                    images = images[valid_mask].to(device)
                    labels = labels[valid_mask].to(device)
                    labels = labels.long()  # Conversion des étiquettes en LongTensor
                    
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    _, preds = torch.max(outputs, 1)
                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    
                except Exception as e:
                    print(f"Erreur lors de la validation: {str(e)}")
                    continue

        val_losses.append(total_val_loss / len(val_loader))
        val_accuracy = accuracy_score(val_labels, val_preds)
        val_accuracies.append(val_accuracy)

        print(f"Époque {epoch+1}, Perte Entraînement: {train_losses[-1]:.4f}, "
              f"Perte Validation: {val_losses[-1]:.4f}, Précision Validation: {val_accuracy:.4f}")

        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            patience_counter = 0
            best_model_state = model.state_dict()
            print("Validation loss improved, modèle sauvegardé.")
        else:
            patience_counter += 1
            print(f"Aucune amélioration de la perte de validation ({patience_counter}/{patience}).")

        if patience_counter >= patience:
            print("Early stopping activé.")
            break

    if best_model_state:
        model.load_state_dict(best_model_state)

    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies
    }

def evaluate_model(model, val_loader, criterion, device):
    model.eval()
    all_predictions = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Évaluation"):
            try:
                valid_mask = labels != -1
                if not valid_mask.any():
                    continue

                images = images[valid_mask].to(device)
                labels = labels[valid_mask].to(device)
                labels = labels.long()  # Conversion des étiquettes en LongTensor
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
                
                _, predictions = torch.max(outputs, 1)
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
            except Exception as e:
                print(f"Erreur lors de l'évaluation: {str(e)}")
                continue
    
    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    
    print("\n=== Résultats de l'évaluation ===")
    print(f"Perte moyenne: {avg_loss:.4f}")
    print(f"Précision globale: {accuracy:.4f}")
    
    print("\nRapport de classification:")
    print(classification_report(all_labels, all_predictions, labels=[0, 1, 2, 3]))
    
    conf_matrix = confusion_matrix(all_labels, all_predictions, labels=[0, 1, 2, 3])
    plt.figure(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix,
                                 display_labels=['Classe ' + str(i) for i in range(4)])
    disp.plot(cmap='Blues')
    plt.title("Matrice de Confusion")
    plt.show()
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'predictions': all_predictions,
        'true_labels': all_labels,
        'confusion_matrix': conf_matrix
    }

def plot_training_metrics(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history['train_losses'], label='Train Loss')
    ax1.plot(history['val_losses'], label='Validation Loss')
    ax1.set_title('Évolution des pertes')
    ax1.set_xlabel('Époque')
    ax1.set_ylabel('Perte')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history['train_accuracies'], label='Train Accuracy')
    ax2.plot(history['val_accuracies'], label='Validation Accuracy')
    ax2.set_title('Évolution des précisions')
    ax2.set_xlabel('Époque')
    ax2.set_ylabel('Précision')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Programme principal
print("Chargement des données...")
df = pd.read_csv(csv_path)

# Vérification des images existantes
existing_images = set(os.listdir(image_folder))
df = df[df['Image_Name'].isin(existing_images)]
df['class'] = df['class'].astype(int)

# Division train/validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Affichage des distributions
plot_class_distribution(train_df, "Distribution des classes dans l'ensemble d'entraînement")
plot_class_distribution(val_df, "Distribution des classes dans l'ensemble de validation")

count_classes(train_df, "Ensemble d'entraînement")
count_classes(val_df, "Ensemble de validation")

# Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Création des datasets et dataloaders
print("Préparation des données...")
train_dataset = ImageDataset(train_df, image_folder, transform)
val_dataset = ImageDataset(val_df, image_folder, transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Configuration du device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device: {device}")

# Initialisation du modèle
print("Initialisation du modèle...")
model = VGG16Standard(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

# Entraînement
print("Début de l'entraînement...")
model, history = train_model_with_early_stopping(
    model, train_loader, val_loader, criterion, optimizer, 
    num_epochs, patience, device
)

# Visualisation des métriques
print("Visualisation des métriques d'entraînement...")
plot_training_metrics(history)

# Évaluation finale
print("Évaluation finale du modèle...")
evaluation_results = evaluate_model(model, val_loader, criterion, device)

# Sauvegarde du modèle et des métriques
print("Sauvegarde du modèle...")
model_name = f"model_vgg16_fullyConnected_sans_augm_couche_nongelee_batch_{batch_size}.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'evaluation_results': evaluation_results
}, os.path.join(models_path, model_name))
print(f"Modèle et métriques sauvegardés sous : {model_name}")
print("Terminé!")